In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from tqdm import tqdm
import os

print("Libraries imported successfully.")

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
df.dropna(axis = 1, inplace = True)
df

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

print(label_encoders)
df

#no categorical values

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

#the target is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
%pip install kagglehub catboost lightgbm tqdm -q
from catboost import CatBoostRegressor

In [ ]:
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
models = {
  "CatBoost": CatBoostRegressor(verbose=0)
}

In [ ]:
lr_accuracy = []
lr_f1 = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    for model_name, model in models.items():
      print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred = (y_pred >= 0.5).astype(int) #h the target is 1 or 0


    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_f1.append(f1)

In [ ]:
print("Accuracy: ", lr_accuracy)
print("F1-Score", lr_f1)

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0, max_iter=10000)
lasso = Lasso(alpha=1.0,  max_iter=10000)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = lasso.coef_
coeffs['Ridge'] = ridge.coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: